# EXP-28: Bold — Pseudo-Labeling + Feature Expansion + 9 Models

**Competition:** SPR 2026 — Mammography Report Classification  
**Base:** EXP-24 (0.81021 LB) — Specialist Ensemble  
**Strategy:** Ousado/Bold — várias melhorias simultâneas  

**Melhorias sobre EXP-24:**
1. **Pseudo-labeling**: Usa predições de alta confiança no test para expandir treino
2. **+2 modelos**: ExtraTrees + SVC com seed diferente para diversidade
3. **TF-IDF achados char 3-6**: Expansão de char ngrams no achados
4. **Label cleaning iterativo**: Corrige labels (não só remove) por 2 iterações
5. **Conclusion features mais ricas**: bigram+trigram na conclusão
6. **Threshold search mais fino**: step 0.005 ao invés de 0.01

**Risco:** Pseudo-labeling pode amplificar erros. Label correction pode mudar distribuição.  
**Runtime:** CPU only

In [ ]:
import os, re, hashlib, warnings, time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from scipy.sparse import hstack, csr_matrix, vstack
import lightgbm as lgb
warnings.filterwarnings('ignore')

TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

N_FOLDS = 5
N_CLASSES = 7

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Class distribution:\n{train["target"].value_counts().sort_index()}')

In [ ]:
# =============================================================================
# Cell 2: Text Preprocessing (igual EXP-24)
# =============================================================================

def clean_achados(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    match = re.search(r'achados[:\s]*(.*?)(?:impress[aã]o|conclus[aã]o|an[aá]lise comparativa|opini[aã]o|$)', s, flags=re.DOTALL)
    if match: s = match.group(1).strip()
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def clean_full(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def extract_conclusion(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    match = re.search(r'(?:impress[aã]o|conclus[aã]o|opini[aã]o)[:\s]*(.*?)(?:an[aá]lise comparativa|recomenda|$)', s, flags=re.DOTALL)
    if match:
        c = match.group(1).strip()
        c = re.sub(r'[0-9]+,[0-9]+', 'NUM', c)
        c = re.sub(r'[0-9]+', 'NUM', c)
        return c
    return ""

def stable_hash(s):
    return hashlib.md5(s.encode("utf-8")).hexdigest()

for df in [train, test]:
    df["achados"]    = df["report"].apply(clean_achados)
    df["full"]       = df["report"].apply(clean_full)
    df["conclusion"] = df["report"].apply(extract_conclusion)
train["group"] = train["report"].apply(stable_hash)

y = train["target"].astype(int).values
groups = train["group"].values

print(f'Achados found: {(train["achados"] != "").sum()} / {len(train)}')
print(f'Conclusion found: {(train["conclusion"] != "").sum()} / {len(train)}')

In [ ]:
# =============================================================================
# Cell 3: Dense Features (igual EXP-24)
# =============================================================================

def extract_birads_number(text):
    match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    return int(match.group(1)) if match else -1

def extract_dense_features(df):
    feat = pd.DataFrame(index=df.index)
    text_col = df['report'].fillna('').astype(str).str.lower()
    
    feat['report_length']   = text_col.apply(len)
    feat['has_measurement'] = text_col.str.contains(r'\b(?:cm|mm|medindo)\b', regex=True).astype(int)
    feat['has_spiculation'] = text_col.str.contains(r'espiculad', regex=True).astype(int)
    feat['has_distortion']  = text_col.str.contains(r'distor[cç][aã]o arquitetural', regex=True).astype(int)
    feat['has_biopsy']      = text_col.str.contains(r'biopsy|bi[oó]psia|resultado de cine|carcinoma', regex=True).astype(int)
    feat['word_count']        = text_col.str.split().str.len().fillna(0).astype(int)
    feat['line_count']        = text_col.str.count('\n') + 1
    feat['achados_length']    = df['achados'].str.len() if 'achados' in df.columns else 0
    feat['conclusion_length'] = df['conclusion'].str.len() if 'conclusion' in df.columns else 0
    feat['has_nodule']        = text_col.str.contains(r'n[oó]dulo', regex=True).astype(int)
    feat['has_calcification'] = text_col.str.contains(r'calcifica[cç]', regex=True).astype(int)
    feat['has_microcalc']     = text_col.str.contains(r'microcalcifica', regex=True).astype(int)
    feat['has_asymmetry']     = text_col.str.contains(r'assimetria', regex=True).astype(int)
    feat['has_lymphnode']     = text_col.str.contains(r'linfonodo|axilar', regex=True).astype(int)
    feat['has_suspicious']    = text_col.str.contains(r'suspeit[oa]|malign[oa]', regex=True).astype(int)
    feat['has_benign']        = text_col.str.contains(r'benign[oa]|cisto[s]?\b', regex=True).astype(int)
    feat['has_gross_calc']    = text_col.str.contains(r'calcifica[cç][aã]o grosseira|calcifica[cç][aã]o vascular', regex=True).astype(int)
    feat['has_nodulo_espic']  = text_col.str.contains(r'n[oó]dulo\s+espiculad', regex=True).astype(int)
    feat['has_heterogeneo']   = text_col.str.contains(r'heterog[eê]ne', regex=True).astype(int)
    feat['has_irregular']     = text_col.str.contains(r'irregular', regex=True).astype(int)
    feat['has_bilateral']     = text_col.str.contains(r'bilateral', regex=True).astype(int)
    feat['has_birads_explicit'] = text_col.str.contains(r'bi-?rads|categoria\s*\d', regex=True).astype(int)
    feat['birads_mentioned']    = text_col.apply(extract_birads_number)
    feat['risk_score'] = (
        feat['has_spiculation'] * 3 + feat['has_distortion'] * 3 +
        feat['has_nodulo_espic'] * 4 + feat['has_biopsy'] * 5 +
        feat['has_suspicious'] * 3 + feat['has_irregular'] * 2 +
        feat['has_microcalc'] * 2 + feat['has_asymmetry'] * 2 -
        feat['has_benign'] * 2 - feat['has_gross_calc'] * 2
    )
    return csr_matrix(feat.values.astype(np.float32))

X_train_dense = extract_dense_features(train)
X_test_dense  = extract_dense_features(test)
print(f'Dense features: {X_train_dense.shape[1]}')

In [ ]:
# =============================================================================
# Cell 4: TF-IDF Features (EXP-24 + extras)
# =============================================================================

print("Building TF-IDF features...")

# Achados — EXPANDED char ngrams (3-6 instead of 3-5)
tfidf_A = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=100000))
])
X_train_A = tfidf_A.fit_transform(train["achados"].values)
X_test_A  = tfidf_A.transform(test["achados"].values)

# Full text config 1 (igual EXP-24)
tfidf_F = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_F = tfidf_F.fit_transform(train["full"].values)
X_test_F  = tfidf_F.transform(test["full"].values)

# Full text config 2 (igual EXP-24)
tfidf_F2 = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=100000))
])
X_train_F2 = tfidf_F2.fit_transform(train["full"].values)
X_test_F2  = tfidf_F2.transform(test["full"].values)

# Conclusion — EXPANDED to trigrams
tfidf_C = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_df=0.95,
                             sublinear_tf=True, max_features=50000))
])
X_train_C = tfidf_C.fit_transform(train["conclusion"].values)
X_test_C  = tfidf_C.transform(test["conclusion"].values)

X_train_lgb = hstack([X_train_A, X_train_dense]).tocsr()
X_test_lgb  = hstack([X_test_A,  X_test_dense]).tocsr()

X_train_full_dense = hstack([X_train_F, X_train_C, X_train_dense]).tocsr()
X_test_full_dense  = hstack([X_test_F,  X_test_C, X_test_dense]).tocsr()

# NOVO: ExtraTrees usa achados + conclusion + dense
X_train_et = hstack([X_train_A, X_train_C, X_train_dense]).tocsr()
X_test_et  = hstack([X_test_A,  X_test_C, X_test_dense]).tocsr()

print(f"SVC-A: {X_train_A.shape[1]}, SVC-F: {X_train_F.shape[1]}, SVC-F2: {X_train_F2.shape[1]}")
print(f"Conclusion: {X_train_C.shape[1]}, LGB: {X_train_lgb.shape[1]}")
print(f"ExtraTrees: {X_train_et.shape[1]}, Full+Dense: {X_train_full_dense.shape[1]}")

In [ ]:
# =============================================================================
# Cell 5: PASS 1 — Run EXP-24 pipeline to get OOF + test predictions
# =============================================================================

def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(X_train_A, y, groups))

L1_MODELS_P1 = [
    ('svc_A', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000), cv=3, method='sigmoid'),
     X_train_A, X_test_A),
    ('svc_F', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000), cv=3, method='sigmoid'),
     X_train_F, X_test_F),
    ('svc_F2', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000), cv=3, method='sigmoid'),
     X_train_F2, X_test_F2),
    ('lgb1', lambda: lgb.LGBMClassifier(
        class_weight='balanced', n_estimators=300, learning_rate=0.05,
        max_depth=6, random_state=42, n_jobs=-1, verbose=-1),
     X_train_lgb, X_test_lgb),
]

print("=== PASS 1: Getting predictions for label cleaning + pseudo-labeling ===")
oof_probas_p1 = {}
test_probas_p1 = {}
t0 = time.time()
for name, model_fn, X_tr, X_te in L1_MODELS_P1:
    print(f"\n--- {name} ---")
    oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds):
        m = model_fn()
        m.fit(X_tr[tr_idx], y[tr_idx])
        if hasattr(m, 'predict_proba'):
            oof[va_idx] = m.predict_proba(X_tr[va_idx])
            te_acc += m.predict_proba(X_te) / N_FOLDS
        else:
            oof[va_idx] = softmax(m.decision_function(X_tr[va_idx]))
            te_acc += softmax(m.decision_function(X_te)) / N_FOLDS
    oof_f1 = f1_score(y, np.argmax(oof, axis=1), average='macro')
    print(f"  OOF F1: {oof_f1:.4f}")
    oof_probas_p1[name] = oof
    test_probas_p1[name] = te_acc

print(f"\nPass 1 done in {time.time()-t0:.0f}s")

In [ ]:
# =============================================================================
# Cell 6: Label Cleaning — Iterative correction (2 passes)
# Corrige labels quando todos modelos concordam na alternativa (P<0.05)
# Remove quando discordam entre si (P<0.03)
# =============================================================================

CORRECT_THRESHOLD = 0.05
REMOVE_THRESHOLD = 0.03

# Blend P1 OOF
oof_blend_p1 = np.mean([oof_probas_p1[name] for name in oof_probas_p1], axis=0)
oof_label_prob = oof_blend_p1[np.arange(len(y)), y]

model_preds = {name: np.argmax(oof, axis=1) for name, oof in oof_probas_p1.items()}
alt_classes = np.stack([model_preds[name] for name in model_preds], axis=1)
all_disagree = np.all(alt_classes != y[:, None], axis=1)
all_agree_alt = np.all(alt_classes == alt_classes[:, 0:1], axis=1)

# Correct: P(label) < 0.05, all models disagree, all agree on same alt
correct_mask = (oof_label_prob < CORRECT_THRESHOLD) & all_disagree & all_agree_alt
# Remove: P(label) < 0.03, all models disagree, but DON'T agree on alt (ambiguous)
remove_mask = (oof_label_prob < REMOVE_THRESHOLD) & all_disagree & ~all_agree_alt

n_correct = correct_mask.sum()
n_remove = remove_mask.sum()
print(f"Labels to CORRECT: {n_correct}")
print(f"Labels to REMOVE:  {n_remove}")

# Apply corrections
y_cleaned = y.copy()
if n_correct > 0:
    new_labels = np.argmax(oof_blend_p1[correct_mask], axis=1)
    print(f"\nCorrections:")
    for old, new in zip(y[correct_mask], new_labels):
        print(f"  {old} → {new}")
    y_cleaned[correct_mask] = new_labels

# Remove ambiguous
keep_mask = ~remove_mask
train_clean = train[keep_mask].reset_index(drop=True)
y_cleaned = y_cleaned[keep_mask]
groups_clean = groups[keep_mask]

print(f"\nClean train: {len(train_clean)} (corrected {n_correct}, removed {n_remove})")
print(f"Class distribution after cleaning:\n{pd.Series(y_cleaned).value_counts().sort_index()}")

In [ ]:
# =============================================================================
# Cell 7: Pseudo-Labeling — Add high-confidence test samples to train
# =============================================================================

PSEUDO_CONFIDENCE = 0.90  # Only add test samples with P(class) > 90%

# Blend P1 test predictions
te_blend_p1 = np.mean([test_probas_p1[name] for name in test_probas_p1], axis=0)
te_max_prob = te_blend_p1.max(axis=1)
te_pred = np.argmax(te_blend_p1, axis=1)

pseudo_mask = te_max_prob >= PSEUDO_CONFIDENCE
n_pseudo = pseudo_mask.sum()
print(f"Test samples with P >= {PSEUDO_CONFIDENCE}: {n_pseudo} / {len(test)} ({100*n_pseudo/len(test):.1f}%)")
print(f"Pseudo-label class distribution:")
print(pd.Series(te_pred[pseudo_mask]).value_counts().sort_index())

# Create pseudo-labeled dataframe
test_pseudo = test[pseudo_mask].copy().reset_index(drop=True)
test_pseudo['target'] = te_pred[pseudo_mask]
test_pseudo['group'] = test_pseudo['report'].apply(stable_hash).apply(lambda x: 'pseudo_' + x)

# Combine clean train + pseudo-labeled test
train_combined = pd.concat([train_clean, test_pseudo[['report', 'achados', 'full', 'conclusion', 'group', 'target']]], 
                           ignore_index=True)
y_combined = train_combined['target'].astype(int).values
groups_combined = train_combined['group'].values

print(f"\nCombined train: {len(train_combined)} (clean: {len(train_clean)}, pseudo: {n_pseudo})")
print(f"Combined class distribution:\n{pd.Series(y_combined).value_counts().sort_index()}")

In [ ]:
# =============================================================================
# Cell 8: Rebuild features on combined data
# =============================================================================

print("Rebuilding TF-IDF on combined data...")

tfidf_A2 = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=100000))
])
Xc_train_A = tfidf_A2.fit_transform(train_combined["achados"].values)
Xc_test_A  = tfidf_A2.transform(test["achados"].values)

tfidf_F_2 = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
Xc_train_F = tfidf_F_2.fit_transform(train_combined["full"].values)
Xc_test_F  = tfidf_F_2.transform(test["full"].values)

tfidf_F2_2 = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=100000))
])
Xc_train_F2 = tfidf_F2_2.fit_transform(train_combined["full"].values)
Xc_test_F2  = tfidf_F2_2.transform(test["full"].values)

tfidf_C2 = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_df=0.95,
                             sublinear_tf=True, max_features=50000))
])
Xc_train_C = tfidf_C2.fit_transform(train_combined["conclusion"].values)
Xc_test_C  = tfidf_C2.transform(test["conclusion"].values)

Xc_train_dense = extract_dense_features(train_combined)
Xc_test_dense  = X_test_dense

Xc_train_lgb = hstack([Xc_train_A, Xc_train_dense]).tocsr()
Xc_test_lgb  = hstack([Xc_test_A,  Xc_test_dense]).tocsr()

Xc_train_full_dense = hstack([Xc_train_F, Xc_train_C, Xc_train_dense]).tocsr()
Xc_test_full_dense  = hstack([Xc_test_F,  Xc_test_C, Xc_test_dense]).tocsr()

Xc_train_et = hstack([Xc_train_A, Xc_train_C, Xc_train_dense]).tocsr()
Xc_test_et  = hstack([Xc_test_A,  Xc_test_C, Xc_test_dense]).tocsr()

print("Features rebuilt on combined data.")

In [ ]:
# =============================================================================
# Cell 9: PASS 2 — Level 1 with 9 models on combined data
# =============================================================================

gkf2 = GroupKFold(n_splits=N_FOLDS)
folds2 = list(gkf2.split(Xc_train_A, y_combined, groups_combined))

# 9 models: EXP-24's 7 + ExtraTrees + SVC seed 123
L1_MODELS = [
    ('svc_A', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000), cv=3, method='sigmoid'),
     Xc_train_A, Xc_test_A),
    ('svc_F', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000), cv=3, method='sigmoid'),
     Xc_train_F, Xc_test_F),
    ('svc_F2', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000), cv=3, method='sigmoid'),
     Xc_train_F2, Xc_test_F2),
    # NOVO: SVC com seed diferente para diversidade
    ('svc_F_s2', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", C=0.8, random_state=123, max_iter=10000), cv=3, method='sigmoid'),
     Xc_train_F, Xc_test_F),
    ('lgb1', lambda: lgb.LGBMClassifier(
        class_weight='balanced', n_estimators=300, learning_rate=0.05,
        max_depth=6, random_state=42, n_jobs=-1, verbose=-1),
     Xc_train_lgb, Xc_test_lgb),
    ('lgb2', lambda: lgb.LGBMClassifier(
        class_weight='balanced', n_estimators=500, learning_rate=0.03,
        max_depth=7, num_leaves=63, min_child_samples=20,
        subsample=0.8, colsample_bytree=0.6,
        random_state=123, n_jobs=-1, verbose=-1),
     Xc_train_lgb, Xc_test_lgb),
    ('lr', lambda: LogisticRegression(
        class_weight='balanced', C=1.0, max_iter=5000,
        solver='lbfgs', multi_class='multinomial', random_state=42, n_jobs=-1),
     Xc_train_full_dense, Xc_test_full_dense),
    ('ridge', lambda: RidgeClassifier(class_weight='balanced', alpha=1.0),
     Xc_train_F, Xc_test_F),
    # NOVO: ExtraTrees para diversidade de algoritmo
    ('et', lambda: ExtraTreesClassifier(
        n_estimators=500, max_depth=None, min_samples_leaf=2,
        class_weight='balanced', random_state=42, n_jobs=-1),
     Xc_train_et, Xc_test_et),
]

print(f"=== PASS 2: Training {len(L1_MODELS)} models on combined data ===")
oof_probas = {}
test_probas = {}
t0 = time.time()
for name, model_fn, X_tr, X_te in L1_MODELS:
    print(f"\n--- Level 1: {name} ---")
    oof = np.zeros((len(train_combined), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds2):
        m = model_fn()
        m.fit(X_tr[tr_idx], y_combined[tr_idx])
        if hasattr(m, 'predict_proba'):
            oof[va_idx] = m.predict_proba(X_tr[va_idx])
            te_acc += m.predict_proba(X_te) / N_FOLDS
        else:
            oof[va_idx] = softmax(m.decision_function(X_tr[va_idx]))
            te_acc += softmax(m.decision_function(X_te)) / N_FOLDS
        fold_f1 = f1_score(y_combined[va_idx], np.argmax(oof[va_idx], axis=1), average='macro')
        print(f"  Fold {fi}: F1={fold_f1:.4f}")
    oof_f1 = f1_score(y_combined, np.argmax(oof, axis=1), average='macro')
    print(f"  >> OOF F1: {oof_f1:.4f}")
    oof_probas[name] = oof
    test_probas[name] = te_acc

print(f"\nLevel 1 done: {len(L1_MODELS)} models in {time.time()-t0:.0f}s")

In [ ]:
# =============================================================================
# Cell 10: Level 2 — Meta-Learner
# =============================================================================

model_names = list(oof_probas.keys())
X_meta_train = np.hstack([oof_probas[name] for name in model_names])
X_meta_test  = np.hstack([test_probas[name] for name in model_names])
print(f"Meta-features: {X_meta_train.shape[1]} ({len(model_names)} models x {N_CLASSES} classes)")

# Test multiple C values
best_meta_f1 = 0
best_meta_C = None
best_meta_oof = None
best_meta_test = None

for C_val in [1.0, 0.5, 0.3]:
    meta_oof = np.zeros((len(train_combined), N_CLASSES), dtype=np.float64)
    meta_test_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds2):
        meta_lr = LogisticRegression(class_weight='balanced', C=C_val, max_iter=5000,
                                     solver='lbfgs', multi_class='multinomial', random_state=42)
        meta_lr.fit(X_meta_train[tr_idx], y_combined[tr_idx])
        meta_oof[va_idx] = meta_lr.predict_proba(X_meta_train[va_idx])
        meta_test_acc += meta_lr.predict_proba(X_meta_test) / N_FOLDS
    meta_f1 = f1_score(y_combined, np.argmax(meta_oof, axis=1), average='macro')
    print(f"Meta C={C_val}: OOF F1={meta_f1:.4f}")
    if meta_f1 > best_meta_f1:
        best_meta_f1 = meta_f1
        best_meta_C = C_val
        best_meta_oof = meta_oof.copy()
        best_meta_test = meta_test_acc.copy()

print(f"\nBest meta C={best_meta_C}: F1={best_meta_f1:.4f}")

# V12-style weighted avg (original 4 SVCs + LGB)
oof_svc_ens = 0.20 * oof_probas['svc_A'] + 0.30 * oof_probas['svc_F'] + 0.25 * oof_probas['svc_F2'] + 0.25 * oof_probas['svc_F_s2']
oof_v12 = 0.65 * oof_svc_ens + 0.35 * oof_probas['lgb1']
v12_f1 = f1_score(y_combined, np.argmax(oof_v12, axis=1), average='macro')
print(f"V12-style OOF: {v12_f1:.4f}")

# Hybrid
oof_hybrid = 0.5 * best_meta_oof + 0.5 * oof_v12
hybrid_f1 = f1_score(y_combined, np.argmax(oof_hybrid, axis=1), average='macro')
print(f"Hybrid OOF: {hybrid_f1:.4f}")

# Select best
results = {'meta': (best_meta_f1, best_meta_oof, best_meta_test)}
te_svc_ens = 0.20 * test_probas['svc_A'] + 0.30 * test_probas['svc_F'] + 0.25 * test_probas['svc_F2'] + 0.25 * test_probas['svc_F_s2']
te_v12 = 0.65 * te_svc_ens + 0.35 * test_probas['lgb1']
results['v12'] = (v12_f1, oof_v12, te_v12)
results['hybrid'] = (hybrid_f1, oof_hybrid, 0.5 * best_meta_test + 0.5 * te_v12)

best = max(results, key=lambda k: results[k][0])
final_score, oof_blend, te_blend = results[best]
print(f"\nBest: {best} (F1={final_score:.4f})")

In [ ]:
# =============================================================================
# Cell 11: Threshold Search FINO (step 0.005)
# =============================================================================

threshold_classes = [6, 5, 4, 3, 1, 0]
threshold_range = np.arange(0.05, 0.55, 0.005)  # FINER step

def apply_thresholds(proba, thresholds):
    preds = np.argmax(proba, axis=1).copy()
    for cls in threshold_classes:
        if cls in thresholds:
            mask = proba[:, cls] > thresholds[cls]
            for higher_cls in threshold_classes:
                if higher_cls == cls: break
                if higher_cls in thresholds:
                    mask = mask & (preds != higher_cls)
            preds[mask] = cls
    return preds

baseline_preds = np.argmax(oof_blend, axis=1)
baseline_f1 = f1_score(y_combined, baseline_preds, average='macro')
print(f'Baseline OOF (no thresholds): {baseline_f1:.4f}')

best_thresholds = {}
current_f1 = baseline_f1
for cls in threshold_classes:
    best_cls_f1 = current_f1
    best_cls_thr = None
    for thr in threshold_range:
        trial = {**best_thresholds, cls: thr}
        trial_preds = apply_thresholds(oof_blend, trial)
        trial_f1 = f1_score(y_combined, trial_preds, average='macro')
        if trial_f1 > best_cls_f1:
            best_cls_f1 = trial_f1
            best_cls_thr = thr
    if best_cls_thr is not None:
        best_thresholds[cls] = best_cls_thr
        current_f1 = best_cls_f1
        print(f'Class {cls}: threshold={best_cls_thr:.3f}, F1={best_cls_f1:.4f}')
    else:
        print(f'Class {cls}: no improvement')

final_oof_preds = apply_thresholds(oof_blend, best_thresholds)
final_oof_f1 = f1_score(y_combined, final_oof_preds, average='macro')
print(f'\nFinal OOF with thresholds: {final_oof_f1:.4f}')
print(f'Thresholds: {best_thresholds}')
print(classification_report(y_combined, final_oof_preds, digits=4))

In [ ]:
# =============================================================================
# Cell 12: Clinical Guardrails + Submission
# =============================================================================

test_preds = apply_thresholds(te_blend, best_thresholds)
test['target'] = test_preds

def apply_clinical_guardrails(row):
    text = str(row['report']).lower()
    pred = int(row['target'])
    
    birads_match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    if birads_match:
        mentioned = int(birads_match.group(1))
        if 0 <= mentioned <= 6:
            return mentioned
    
    if re.search(r'resultado de cine grau 3|carcinoma|\bcdis\b|neoplasia maligna', text):
        return 6
    
    if re.search(r'n[oó]dulo\s+espiculad', text) and pred < 4:
        return 5
    if 'espiculad' in text and 'distor' in text and pred < 4:
        return 5
    
    if re.search(r'calcifica[cç][aã]o grosseira|calcifica[cç][aã]o vascular', text):
        if pred >= 4 and not re.search(r'espiculad|suspeit|malign|distor', text):
            return 2
    
    return pred

test['target'] = test.apply(apply_clinical_guardrails, axis=1)

submission = test[['ID', 'target']].copy()
submission['target'] = submission['target'].astype(int)
submission.to_csv('submission.csv', index=False)

print('Prediction distribution:')
print(submission['target'].value_counts().sort_index())
print(f'\nSubmission saved! Shape: {submission.shape}')
print(submission)